# Reshaping Data in Pandas

Reshaping means **changing the structure of a DataFrame without changing the underlying information**.

We will cover:

1. `pivot()`
2. `pivot_table()`
3. `melt()`
4. Difference between `pivot()` and `pivot_table()`
5. Real-world example

---

## 11.1 `pivot()`

`pivot()` converts **unique row values into columns**.

### Example

In [11]:
import pandas as pd

df = pd.DataFrame({
    "Month": ["Jan", "Jan", "Feb", "Feb"],
    "Product": ["Laptop", "Mobile", "Laptop", "Mobile"],
    "Sales": [50000, 30000, 60000, 35000]
})
# df.drop_duplicates(inplace= True)
df

,Month,Product,Sales
0,Jan,Laptop,50000
1,Jan,Mobile,30000
2,Feb,Laptop,60000
3,Feb,Mobile,35000


Output:

```text
  Month Product  Sales
0   Jan  Laptop  50000
1   Jan  Mobile  30000
2   Feb  Laptop  60000
3   Feb  Mobile  35000
```

Now reshape it:

In [12]:
result = df.pivot(
    index="Month",
    columns="Product",
    values="Sales"
)

result

Product,Laptop,Mobile
Month,,
Feb,60000,35000
Jan,50000,30000


Output:

```text
Product  Laptop  Mobile
Month
Feb       60000   35000
Jan       50000   30000
```

### Structure

```text
pivot(
    index=,
    columns=,
    values=
)
```

Think of it as:

```text
Rows      → index
Columns   → columns
Values    → values
```

---

# 11.2 `pivot_table()`

`pivot_table()` is similar to `pivot()`, but it can **aggregate duplicate records**.

This is extremely useful in real-world data.

In [14]:
df = pd.DataFrame({
    "Month": ["Jan", "Jan", "Jan", "Feb", "Feb"],
    "Product": ["Laptop", "Laptop", "Mobile", "Laptop", "Mobile"],
    "Sales": [50000, 55000, 30000, 60000, 35000]
})
df

,Month,Product,Sales
0,Jan,Laptop,50000
1,Jan,Laptop,55000
2,Jan,Mobile,30000
3,Feb,Laptop,60000
4,Feb,Mobile,35000


In [18]:
df.groupby(by = ["Month", "Product"]).agg( 
    total_sales = ("Sales", "sum") 
)

total_sales
Month Product             
Feb   Laptop         60000
      Mobile         35000
Jan   Laptop        105000
      Mobile         30000

In [21]:
df.pivot_table(
    index = ["Month"],
    columns= ["Product"],
    aggfunc={
        "Sales" : "sum"
    }
)

Sales       
Product  Laptop Mobile
Month                 
Feb       60000  35000
Jan      105000  30000

Here January has **two Laptop records**.

Using:

In [11]:
# df.pivot(
#     index="Month",
#     columns="Product",
#     values="Sales"
# )
df

,Month,Product,Sales
0,Jan,Laptop,50000
1,Jan,Laptop,55000
2,Jan,Mobile,30000
3,Feb,Laptop,60000
4,Feb,Mobile,35000


will produce an error because the `Month + Product` combination is not unique.

Instead:

In [13]:
result = df.pivot_table(
    index="Month",
    columns="Product",
    values="Sales",
    aggfunc="sum"
)

result

Product,Laptop,Mobile
Month,,
Feb,60000,35000
Jan,105000,30000


In [27]:
products = pd.read_csv('p_data.csv')
products

,Unnamed: 0,S. No.,Category ID,Main Category,Subcategory ID,Subcategory,Item ID,Item Name,Quantity,Unit Price (₹),Discount (%),Total Amount (₹),Rating
0,0,1,101,Electronics,201,Mobiles,1001,Smartphone A,5,"25,000",10,"112,500",4.5
1,1,2,101,Electronics,201,Mobiles,1002,Smartphone B,3,"18,000",5,"51,300",4.2
2,2,3,101,Electronics,202,Laptops,1003,Laptop A,2,"65,000",8,"119,600",4.7
3,3,4,101,Electronics,202,Laptops,1004,Laptop B,4,"55,000",12,"193,600",4.3
4,4,5,102,Clothing,203,Men,1005,Formal Shirt,10,"1,200",15,"10,200",4.1
5,5,6,102,Clothing,203,Men,1006,Jeans,8,"2,000",10,"14,400",4.4
6,6,7,102,Clothing,204,Women,1007,Kurti,12,900,5,"10,260",4.0
7,7,8,102,Clothing,204,Women,1008,Saree,4,"3,500",20,"11,200",4.6
8,8,9,103,Furniture,205,Living Room,1009,Sofa,2,"32,000",10,"57,600",4.8
9,9,10,103,Furniture,205,Living Room,1010,Coffee Table,5,"4,500",5,"21,375",4.2


In [39]:
products.columns

Index(['Unnamed: 0', 'S. No.', 'Category ID', 'Main Category',
       'Subcategory ID', 'Subcategory', 'Item ID', 'Item Name', 'Quantity',
       'Unit Price (₹)', 'Discount (%)', 'Total Amount (₹)', 'Rating'],
      dtype='str')

In [49]:
products['Total Amount (₹)'] = products['Total Amount (₹)'].replace(",", "", regex= True)
products['Total Amount (₹)'] = products['Total Amount (₹)'].astype(dtype='int')
products.info()


<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        20 non-null     int64  
 1   S. No.            20 non-null     int64  
 2   Category ID       20 non-null     int64  
 3   Main Category     20 non-null     str    
 4   Subcategory ID    20 non-null     int64  
 5   Subcategory       20 non-null     str    
 6   Item ID           20 non-null     int64  
 7   Item Name         20 non-null     str    
 8   Quantity          20 non-null     int64  
 9   Unit Price (₹)    20 non-null     str    
 10  Discount (%)      20 non-null     int64  
 11  Total Amount (₹)  20 non-null     int64  
 12  Rating            20 non-null     float64
dtypes: float64(1), int64(8), str(4)
memory usage: 2.7 KB


In [53]:
products.pivot_table(
    index = "Main Category",
    columns = "Subcategory",
    aggfunc={
        "Rating" : "mean"
    }
)

Rating                                                      \
Subcategory   Bedroom Beverages Fitness Laptops Living Room   Men Mobiles   
Main Category                                                               
Clothing          NaN       NaN     NaN     NaN         NaN  4.25     NaN   
Electronics       NaN       NaN     NaN     4.5         NaN   NaN    4.35   
Furniture         4.4       NaN     NaN     NaN         4.5   NaN     NaN   
Grocery           NaN      4.25     NaN     NaN         NaN   NaN     NaN   
Sports            NaN       NaN    4.35     NaN         NaN   NaN     NaN   

                                    
Subcategory   Outdoor Snacks Women  
Main Category                       
Clothing          NaN    NaN   4.3  
Electronics       NaN    NaN   NaN  
Furniture         NaN    NaN   NaN  
Grocery           NaN    4.3   NaN  
Sports           4.45    NaN   NaN

In [52]:
products

,Unnamed: 0,S. No.,Category ID,Main Category,Subcategory ID,Subcategory,Item ID,Item Name,Quantity,Unit Price (₹),Discount (%),Total Amount (₹),Rating
0,0,1,101,Electronics,201,Mobiles,1001,Smartphone A,5,"25,000",10,112500,4.5
1,1,2,101,Electronics,201,Mobiles,1002,Smartphone B,3,"18,000",5,51300,4.2
2,2,3,101,Electronics,202,Laptops,1003,Laptop A,2,"65,000",8,119600,4.7
3,3,4,101,Electronics,202,Laptops,1004,Laptop B,4,"55,000",12,193600,4.3
4,4,5,102,Clothing,203,Men,1005,Formal Shirt,10,"1,200",15,10200,4.1
5,5,6,102,Clothing,203,Men,1006,Jeans,8,"2,000",10,14400,4.4
6,6,7,102,Clothing,204,Women,1007,Kurti,12,900,5,10260,4.0
7,7,8,102,Clothing,204,Women,1008,Saree,4,"3,500",20,11200,4.6
8,8,9,103,Furniture,205,Living Room,1009,Sofa,2,"32,000",10,57600,4.8
9,9,10,103,Furniture,205,Living Room,1010,Coffee Table,5,"4,500",5,21375,4.2


Output:

```text
Product  Laptop  Mobile
Month
Feb       60000   35000
Jan      105000   30000
```

Because:

```text
Jan Laptop
= 50000 + 55000
= 105000
```

---

## 11.3 Different Aggregations

You can use:

In [ ]:
aggfunc="sum"

In [ ]:
aggfunc="mean"

In [ ]:
aggfunc="count"

In [ ]:
aggfunc="max"

Example:

In [14]:
df.pivot_table(
    index="Month",
    columns="Product",
    values="Sales",
    aggfunc="mean"
)

Product,Laptop,Mobile
Month,,
Feb,60000.0,35000.0
Jan,52500.0,30000.0


In [26]:
result = df.pivot_table(
    index="Month",
    values="Sales",
    aggfunc=["sum", "mean"]
)

result.columns = ["Total Sales", "Average Sales"]

result

,Total Sales,Average Sales
Month,,
Feb,95000,47500.0
Jan,135000,45000.0


---

# 11.4 `melt()`

`melt()` does the **opposite type of transformation**.

It converts **wide data → long data**.

Example:

In [54]:
df = pd.DataFrame({
    "Product": ["Laptop", "Mobile", "Tablet"],
    "Jan": [50000, 30000, 20000],
    "Feb": [60000, 35000, 25000],
    "Mar": [70000, 40000, 30000]
})

df

,Product,Jan,Feb,Mar
0,Laptop,50000,60000,70000
1,Mobile,30000,35000,40000
2,Tablet,20000,25000,30000


Output:

```text
  Product    Jan    Feb    Mar
0  Laptop  50000  60000  70000
1  Mobile  30000  35000  40000
2  Tablet  20000  25000  30000
```

Convert it into long format:

In [55]:
result = df.melt(
    id_vars="Product",
    var_name="Month",
    value_name="Sales"
)

result

,Product,Month,Sales
0,Laptop,Jan,50000
1,Mobile,Jan,30000
2,Tablet,Jan,20000
3,Laptop,Feb,60000
4,Mobile,Feb,35000
5,Tablet,Feb,25000
6,Laptop,Mar,70000
7,Mobile,Mar,40000
8,Tablet,Mar,30000


Output:

```text
  Product Month  Sales
0  Laptop   Jan  50000
1  Mobile   Jan  30000
2  Tablet   Jan  20000
3  Laptop   Feb  60000
4  Mobile   Feb  35000
5  Tablet   Feb  25000
6  Laptop   Mar  70000
7  Mobile   Mar  40000
8  Tablet   Mar  30000
```

---

## 11.5 `pivot()` vs `pivot_table()` vs `melt()`

| Function        | Purpose                   |
| --------------- | ------------------------- |
| `pivot()`       | Long → Wide               |
| `pivot_table()` | Long → Wide + aggregation |
| `melt()`        | Wide → Long               |

### Easy way to remember

```text
melt()
   ↓
Wide → Long

pivot()
   ↓
Long → Wide

pivot_table()
   ↓
Long → Wide + aggregation
```

---

# Real-World Example

Suppose a company has sales data:

In [ ]:
sales = pd.DataFrame({
    "Region": ["North", "North", "South", "South"],
    "Product": ["Laptop", "Mobile", "Laptop", "Mobile"],
    "Sales": [100000, 60000, 80000, 50000]
})

Create a report:

In [ ]:
report = sales.pivot_table(
    index="Region",
    columns="Product",
    values="Sales",
    aggfunc="sum"
)

print(report)

Output:

```text
Product  Laptop  Mobile
Region
North    100000   60000
South     80000   50000
```

This type of reshaping is commonly used before creating **Excel reports, dashboards, Power BI datasets, and analytical summaries**.

### Keywords Recap

```text
Reshaping
├── pivot()
├── pivot_table()
├── melt()
├── Wide Data
├── Long Data
└── Aggregation
```